In [0]:
import delta

In [0]:
df_full = spark.read.format("parquet").load("/Volumes/projeto_olist/olist/fullload/customers_fullload/")
df_full.display()
(df_full.coalesce(1)
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("projeto_olist.bronze.customers_fullload"))

In [0]:
df_cdc = spark.read.format("parquet").load("/Volumes/projeto_olist/olist/cdc/customers/")
df_cdc.createOrReplaceTempView("customers")

In [0]:
query = '''

select * from customers

qualify row_number() over (partition by customer_id order by data_particao desc) =1

'''

df_cdc_unique = spark.sql(query)
df_cdc_unique.display()



In [0]:
bronze = delta.DeltaTable.forName(spark, "projeto_olist.bronze.customers_fullload")

In [0]:
(bronze.alias("b")
      .merge(df_cdc_unique.alias("d"), "b.customer_id = d.customer_id")
      .whenMatchedUpdateAll()
      .whenNotMatchedInsertAll()
      .execute())

In [0]:
%sql
select * from projeto_olist.bronze.customers_fullload